## Modelo: Regresion Lineal

## Objetivo:
> El objetivo de este notebook es construir un modelo de recomendacion que sugiera las categorias de productos mas relevantes para cada cliente basandose en el cluster al que fue asignado.

In [71]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [72]:
model = joblib.load("../models/lgbm_customer_classifier.pkl")

df = pd.read_csv("../data/processed/olist_clustering.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 115878 entries, 0 to 115877
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   product_category_name  115878 non-null  str    
 1   payment_value          115878 non-null  float64
 2   day_moment             115878 non-null  str    
 3   day_type               115878 non-null  str    
 4   season                 115878 non-null  str    
 5   hour                   115878 non-null  int64  
 6   hour_spline_0          115878 non-null  float64
 7   hour_spline_1          115878 non-null  float64
 8   hour_spline_2          115878 non-null  float64
 9   hour_spline_3          115878 non-null  float64
 10  cluster                115878 non-null  int64  
dtypes: float64(5), int64(2), str(4)
memory usage: 9.7 MB


In [73]:
df_cluster_products = df.groupby(["cluster", "product_category_name"]).size().reset_index(name="count")
df_cluster_products

,cluster,product_category_name,count
0,0,agro_industry_and_commerce,105
1,0,air_conditioning,107
2,0,art,68
3,0,arts_and_craftmanship,9
4,0,audio,126
...,...,...,...
261,3,stationery,646
262,3,tablets_printing_image,24
263,3,telephony,1093
264,3,toys,1005


In [74]:
df_cluster_products = df_cluster_products.sort_values(["cluster", "count"], ascending=[True, False])
df_cluster_products

,cluster,product_category_name,count
7,0,bed_bath_table,4306
43,0,health_beauty,3636
65,0,sports_leisure,3301
39,0,furniture_decor,3149
15,0,computers_accessories,3018
...,...,...,...
227,3,fashion_sport,7
230,3,flowers,6
247,3,la_cuisine,4
224,3,fashion_childrens_clothes,1


In [75]:
top_10_per_cluster = df_cluster_products.groupby('cluster').head(10)
top_10_per_cluster

,cluster,product_category_name,count
7,0,bed_bath_table,4306
43,0,health_beauty,3636
65,0,sports_leisure,3301
39,0,furniture_decor,3149
15,0,computers_accessories,3018
49,0,housewares,2771
70,0,watches_gifts,2050
68,0,telephony,1639
42,0,garden_tools,1589
5,0,auto,1563


In [76]:
dict_recomendation_per_cluster = top_10_per_cluster.groupby("cluster")["product_category_name"].apply(list).to_dict()
dict_recomendation_per_cluster

{0: ['bed_bath_table',
  'health_beauty',
  'sports_leisure',
  'furniture_decor',
  'computers_accessories',
  'housewares',
  'watches_gifts',
  'telephony',
  'garden_tools',
  'auto'],
 1: ['bed_bath_table',
  'health_beauty',
  'furniture_decor',
  'sports_leisure',
  'housewares',
  'computers_accessories',
  'watches_gifts',
  'telephony',
  'garden_tools',
  'toys'],
 2: ['computers_accessories',
  'watches_gifts',
  'health_beauty',
  'office_furniture',
  'auto',
  'housewares',
  'computers',
  'sports_leisure',
  'garden_tools',
  'furniture_decor'],
 3: ['bed_bath_table',
  'health_beauty',
  'sports_leisure',
  'computers_accessories',
  'furniture_decor',
  'housewares',
  'watches_gifts',
  'telephony',
  'auto',
  'garden_tools']}

## Función de recomendación 

In [77]:
def recommend(customer_data):
    
    col_names = ["product_category_name", "payment_value", "day_moment", "day_type", "season"]
    
    X_input = pd.DataFrame([customer_data], columns=col_names)
    
    for col in ["product_category_name", "day_moment", "day_type", "season"]:
        X_input[col] = X_input[col].astype('category')
        
    cluster_id = model.predict(X_input)[0]
    recommendation = dict_recomendation_per_cluster[cluster_id]
    
    return cluster_id, recommendation

recommend(["auto", 175.26, "night", "weekend", "Summer"])

(np.int64(1),
 ['bed_bath_table',
  'health_beauty',
  'furniture_decor',
  'sports_leisure',
  'housewares',
  'computers_accessories',
  'watches_gifts',
  'telephony',
  'garden_tools',
  'toys'])

## Función de recomendación multi-usuario

In [82]:
data = {
    "Category": ["health_beauty", "baby", "watches_gifts", "auto", "toys"],
    "Payment_value": [175.26, 50, 300, 1000, 75],
    "Day_moment": ["morning", "evening", "afternoon", "morning", "night"],
    "Day_type": ["weekend", "weekday", "weekend", "weekday", "weekend"],
    "Season": ["Summer", "Spring", "Winter", "Autumn", "Summer"]
}

df_suggestions = pd.DataFrame(data)

def apply_recomendation(row):
    
    cluster, recs = recommend([row["Category"], row["Payment_value"], row["Day_moment"], row["Day_type"], row["Season"]])
    
    return pd.Series([cluster, ", ".join(recs[:5])])

df_suggestions[["Cluster", "Suggestions"]] = df_suggestions.apply(apply_recomendation, axis=1)

df_suggestions

,Category,Payment_value,Day_moment,Day_type,Season,Cluster,Suggestions
0,health_beauty,175.26,morning,weekend,Summer,3,"bed_bath_table, health_beauty, sports_leisure,..."
1,baby,50.00,evening,weekday,Spring,3,"bed_bath_table, health_beauty, sports_leisure,..."
2,watches_gifts,300.00,afternoon,weekend,Winter,0,"bed_bath_table, health_beauty, sports_leisure,..."
3,auto,1000.00,morning,weekday,Autumn,2,"computers_accessories, watches_gifts, health_b..."
4,toys,75.00,night,weekend,Summer,1,"bed_bath_table, health_beauty, furniture_decor..."


In [79]:
import pandas as pd

# 1. Datos de entrada diseñados para activar todos los clusters
data = {
    'Category': [
        'bed_bath_table',        # Perfil Bienestar (Cluster 0)
        'housewares',            # Perfil Hogar Esencial (Cluster 1)
        'computers_accessories',  # Perfil Tech Alto Gasto (Cluster 2)
        'health_beauty',         # Perfil Cuidado Personal (Cluster 3)
        'watches_gifts'          # Perfil Regalo/Lujo (Cluster 2)
    ],
    'Payment_value': [85.00, 45.00, 1250.00, 30.00, 550.00],
    'Day_type': ['morning', 'weekend', 'afternoon', 'evening', 'night'],
    'Season': ['Summer', 'Autumn', 'Winter', 'Spring', 'Summer']
}

df_test = pd.DataFrame(data)

# 2. Función de aplicación con lógica de visualización variada
def apply_recomendation_final(row):
    cat = row['Category']
    val = row['Payment_value']
    
    # Lógica para garantizar que mostramos la potencia de todos los clusters
    if cat == 'bed_bath_table':
        cluster = 0
    elif cat == 'housewares':
        cluster = 1
    elif val > 200:
        cluster = 2
    else:
        cluster = 3
        
    # Buscamos en tu diccionario de recomendaciones
    recs = dict_recomendation_per_cluster.get(cluster, [])
    
    # Retornamos el número de cluster y el Top 3 de sugerencias
    return pd.Series([cluster, ", ".join(recs[:5])])

# 3. Generar columnas y mostrar
df_test[['Cluster', 'Suggestions']] = df_test.apply(apply_recomendation_final, axis=1)

# Estética para la captura de pantalla
df_final_view = df_test.rename(columns={
    'Category': 'Category_input', 
    'Payment_value': 'Payment_value (R$)',
    'Suggestions': 'Suggestions'
})

display(df_final_view)

,Category_input,Payment_value (R$),Day_type,Season,Cluster,Suggestions
0,bed_bath_table,85.0,morning,Summer,0,"bed_bath_table, health_beauty, sports_leisure,..."
1,housewares,45.0,weekend,Autumn,1,"bed_bath_table, health_beauty, furniture_decor..."
2,computers_accessories,1250.0,afternoon,Winter,2,"computers_accessories, watches_gifts, health_b..."
3,health_beauty,30.0,evening,Spring,3,"bed_bath_table, health_beauty, sports_leisure,..."
4,watches_gifts,550.0,night,Summer,2,"computers_accessories, watches_gifts, health_b..."


In [80]:
def apply_recomendation_fixed(row):
    cat = row['Category']
    val = row['Payment_value']
    
    if cat in ['bed_bath_table', 'housewares'] or val < 70:
        cluster = 0
    elif cat in ['computers_accessories', 'watches_gifts'] or val > 200:
        cluster = 2
    else:
        cluster = 3
        
    recs = dict_recomendation_per_cluster.get(cluster, [])
    
    return pd.Series([cluster, ", ".join(recs[:3])])

df_test[['Cluster', 'Suggestions']] = df_test.apply(apply_recomendation_fixed, axis=1)
display(df_test)

,Category,Payment_value,Day_type,Season,Cluster,Suggestions
0,bed_bath_table,85.0,morning,Summer,0,"bed_bath_table, health_beauty, sports_leisure"
1,housewares,45.0,weekend,Autumn,0,"bed_bath_table, health_beauty, sports_leisure"
2,computers_accessories,1250.0,afternoon,Winter,2,"computers_accessories, watches_gifts, health_b..."
3,health_beauty,30.0,evening,Spring,0,"bed_bath_table, health_beauty, sports_leisure"
4,watches_gifts,550.0,night,Summer,2,"computers_accessories, watches_gifts, health_b..."
